### Code dạng “bộ khung” để có thể:

- Lọc COCO và UA-DETRAC theo 6 lớp: ['car', 'truck', 'bus', 'motor', 'bicycle', 'person']
- Convert annotation sang YOLO txt (normalized x_center, y_center, w, h).
Bạn cần chỉnh lại các đường dẫn thư mục cho đúng với máy của bạn rồi chạy.

### 0. Quy ước class & mapping
#### 0.1. Danh sách class mục tiêu

In [1]:
# %% [markdown]
# # 0. Setup & Config

# %%
import os
import json
import random
from pathlib import Path
from collections import defaultdict, Counter
from shutil import copy2

import numpy as np
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

# COCO
COCO_IMG_DIR = Path("D:\Learn\Year4\KLTN\Dataset\COCO2017\\train2017")
COCO_ANN_FILE = Path("D:\Learn\Year4\KLTN\Dataset\COCO2017\\annotations\instances_train2017.json")

# UA-DETRAC
UA_IMG_ROOT   = Path("D:\\Learn\\Year4\\KLTN\\Dataset\\UA-DETRAC_UPD_ANN\\images\\train")
UA_LABEL_ROOT = Path("D:\\Learn\\Year4\\KLTN\\Dataset\\UA-DETRAC_UPD_ANN\\labels\\train")

# Kaggle API
KAGGLE_CONFIG_DIR = "/kaggle/input/kaggle-api"
os.environ["KAGGLE_CONFIG_DIR"] = KAGGLE_CONFIG_DIR

# Dataset hợp nhất
DATA_ROOT = Path("D:\Learn\Year4\KLTN\Dataset\\traffic_yolo")
IMG_ROOT  = DATA_ROOT / "images" / "all"
LBL_ROOT  = DATA_ROOT / "labels" / "all"

IMG_ROOT.mkdir(parents=True, exist_ok=True)
LBL_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_CLASSES = ['car', 'truck', 'bus', 'motor', 'bicycle', 'person']
CLASS_TO_ID = {name: i for i, name in enumerate(TARGET_CLASSES)}
print("TARGET_CLASSES:", TARGET_CLASSES)
print("DATA_ROOT:", DATA_ROOT)

# GIỚI HẠN SỐ ẢNH (bạn chỉnh cho phù hợp)
MAX_COCO_IMAGES = 10000   # ví dụ
MAX_UA_IMAGES   = 15000   # theo yêu cầu ~25–30k


TARGET_CLASSES: ['car', 'truck', 'bus', 'motor', 'bicycle', 'person']
DATA_ROOT: D:\Learn\Year4\KLTN\Dataset\traffic_yolo


In [5]:
# %% [markdown]
# # 1. Convert COCO → YOLO 6 lớp
# - Chỉ giữ các class: car, truck, bus, motorcycle, bicycle, person
# - Convert bbox → YOLO
# - **Không copy ảnh**, thay bằng **symlink** vào DATA_ROOT/images/all (tiết kiệm dung lượng)

# %%
# Mapping tên class COCO → tên class mục tiêu
# --- 1. COCO -> YOLO ---

COCO_TO_TARGET = {
    'car':        'car',
    'truck':      'truck',
    'bus':        'bus',
    'motorcycle': 'motor',
    'bicycle':    'bicycle',
    'person':     'person',
}

def coco_bbox_to_yolo(x, y, w, h, img_w, img_h):
    x_center = x + w / 2.0
    y_center = y + h / 2.0
    return (
        x_center / img_w,
        y_center / img_h,
        w / img_w,
        h / img_h,
    )

print("Loading COCO annotations...")
with open(COCO_ANN_FILE, "r", encoding="utf-8") as f:
    coco = json.load(f)

cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
images_info = {img["id"]: img for img in coco["images"]}

ann_by_image = defaultdict(list)
for ann in coco["annotations"]:
    if ann.get("iscrowd", 0) == 1:
        continue
    cat_name = cat_id_to_name[ann["category_id"]]
    if cat_name not in COCO_TO_TARGET:
        continue
    ann_by_image[ann["image_id"]].append(ann)

print("Ảnh COCO có bbox thuộc 6 lớp:", len(ann_by_image))

valid_img_ids = list(ann_by_image.keys())
random.shuffle(valid_img_ids)

if MAX_COCO_IMAGES > 0:
    valid_img_ids = valid_img_ids[:MAX_COCO_IMAGES]
    print(f"Chỉ dùng {len(valid_img_ids)} ảnh COCO")
else:
    print(f"Dùng toàn bộ {len(valid_img_ids)} ảnh COCO")

coco_bbox_counter = Counter()
coco_img_counter = 0

for img_id in tqdm(valid_img_ids, desc="COCO -> YOLO"):
    img_info = images_info[img_id]
    file_name = img_info["file_name"]
    img_src_path = COCO_IMG_DIR / file_name
    if not img_src_path.exists():
        continue

    img_w = img_info["width"]
    img_h = img_info["height"]

    anns = ann_by_image[img_id]
    lines = []
    for ann in anns:
        cat_name = cat_id_to_name[ann["category_id"]]
        target_name = COCO_TO_TARGET[cat_name]
        cls_id = CLASS_TO_ID[target_name]

        x, y, w, h = ann["bbox"]
        x_n, y_n, w_n, h_n = coco_bbox_to_yolo(x, y, w, h, img_w, img_h)
        if w_n <= 0 or h_n <= 0:
            continue

        lines.append(f"{cls_id} {x_n:.6f} {y_n:.6f} {w_n:.6f} {h_n:.6f}")
        coco_bbox_counter[target_name] += 1

    if not lines:
        continue

    # Copy ảnh (có giới hạn số lượng nên không quá nặng)
    dst_img_path = IMG_ROOT / file_name
    dst_img_path.parent.mkdir(parents=True, exist_ok=True)
    if not dst_img_path.exists():
        copy2(img_src_path, dst_img_path)

    # Ghi label
    dst_lbl_path = LBL_ROOT / (Path(file_name).stem + ".txt")
    with open(dst_lbl_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    coco_img_counter += 1

print("COCO: số ảnh dùng:", coco_img_counter)
print("COCO bbox per class:")
for name in TARGET_CLASSES:
    print(f"{name:8s}: {coco_bbox_counter[name]}")


Loading COCO annotations...
Ảnh COCO có bbox thuộc 6 lớp: 70082
Chỉ dùng 10000 ảnh COCO


COCO -> YOLO: 100%|██████████| 10000/10000 [02:10<00:00, 76.57it/s]

COCO: số ảnh dùng: 10000
COCO bbox per class:
car     : 5821
truck   : 1395
bus     : 906
motor   : 1077
bicycle : 965
person  : 36961


In [6]:
# %% [markdown]
# # 2. Remap UA-DETRAC → 6 lớp & gộp vào dataset
# - Input UA-DETRAC đã là YOLO: old_id x_c y_c w h
# - Mapping:
#   1→bicycle, 2→motor, 3→car, 4→truck, 5→bus, 6→truck, 7→truck, 8/9→bỏ
# - (NEW) Giới hạn số file label UA-DETRAC ~ MAX_UA_IMAGES
# - (NEW) Cũng dùng symlink cho ảnh

# %%
# --- 2. UA-DETRAC -> YOLO 6 lớp ---

UA_KAGGLE_ID_TO_TARGET = {
    1: 'bicycle',
    2: 'motor',
    3: 'car',
    4: 'truck',
    5: 'bus',
    6: 'truck',
    7: 'truck',
    # 8,9 bỏ
}

ua_bbox_counter = Counter()
ua_img_counter = 0

ua_txt_files = sorted(UA_LABEL_ROOT.glob("*.txt"))
print("Tổng label UA-DETRAC:", len(ua_txt_files))

random.shuffle(ua_txt_files)
if MAX_UA_IMAGES > 0 and len(ua_txt_files) > MAX_UA_IMAGES:
    ua_txt_files = ua_txt_files[:MAX_UA_IMAGES]
print("Số label UA-DETRAC sẽ dùng:", len(ua_txt_files))

for src_lbl in tqdm(ua_txt_files, desc="UA-DETRAC -> YOLO"):
    with open(src_lbl, "r", encoding="utf-8") as f:
        lines = f.readlines()

    new_lines = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) != 5:
            continue
        old_id = int(parts[0])
        x_c    = float(parts[1])
        y_c    = float(parts[2])
        w      = float(parts[3])
        h      = float(parts[4])

        if old_id not in UA_KAGGLE_ID_TO_TARGET:
            continue  # 8,9: unknown/mask

        target_name = UA_KAGGLE_ID_TO_TARGET[old_id]
        new_cls_id  = CLASS_TO_ID[target_name]
        new_lines.append(f"{new_cls_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")
        ua_bbox_counter[target_name] += 1

    if not new_lines:
        continue

    stem = src_lbl.stem  # MVI_20011_img00001
    src_img = UA_IMG_ROOT / f"{stem}.jpg"
    if not src_img.exists():
        continue

    dst_img = IMG_ROOT / f"{stem}.jpg"
    dst_img.parent.mkdir(parents=True, exist_ok=True)
    if not dst_img.exists():
        copy2(src_img, dst_img)

    dst_lbl = LBL_ROOT / f"{stem}.txt"
    with open(dst_lbl, "w", encoding="utf-8") as f:
        f.write("\n".join(new_lines))

    ua_img_counter += 1

print("UA-DETRAC: số ảnh dùng:", ua_img_counter)
print("UA bbox per class:")
for name in TARGET_CLASSES:
    print(f"{name:8s}: {ua_bbox_counter[name]}")




Tổng label UA-DETRAC: 83756
Số label UA-DETRAC sẽ dùng: 15000


UA-DETRAC -> YOLO: 100%|██████████| 15000/15000 [06:43<00:00, 37.13it/s]

UA-DETRAC: số ảnh dùng: 14721
UA bbox per class:
car     : 10187
truck   : 6245
bus     : 621
motor   : 86167
bicycle : 1213
person  : 0


In [ ]:
# %% [markdown]
# # 3. Thống kê & tạo train/val + oversample lớp hiếm

# %%
# --- 3. Thống kê & split ---

image_meta = []

for lbl_path in sorted(LBL_ROOT.glob("*.txt")):
    class_counts = Counter()
    with open(lbl_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            if 0 <= cls_id < len(TARGET_CLASSES):
                class_counts[cls_id] += 1

    if not class_counts:
        continue

    img_name = lbl_path.stem + ".jpg"
    img_path = IMG_ROOT / img_name
    if not img_path.exists():
        continue

    image_meta.append({
        "img_path": str(img_path),
        "lbl_path": str(lbl_path),
        "class_counts": class_counts,
    })

print("Tổng ảnh dataset:", len(image_meta))

total_bbox = Counter()
for meta in image_meta:
    for cid, c in meta["class_counts"].items():
        total_bbox[cid] += c

print("Tổng bbox theo lớp:")
for cid, c in total_bbox.items():
    print(f"{TARGET_CLASSES[cid]:8s}: {c}")

random.shuffle(image_meta)
val_ratio = 0.1
val_size = int(len(image_meta) * val_ratio)
val_meta = image_meta[:val_size]
train_meta = image_meta[val_size:]

print(f"Số ảnh train: {len(train_meta)}, val: {len(val_meta)}")

MOTOR_ID   = TARGET_CLASSES.index("motor")
BICYCLE_ID = TARGET_CLASSES.index("bicycle")
BUS_ID     = TARGET_CLASSES.index("bus")

train_list = []
for meta in train_meta:
    cls_cnt = meta["class_counts"]
    img_path = meta["img_path"]

    has_motor   = cls_cnt[MOTOR_ID] > 0
    has_bicycle = cls_cnt[BICYCLE_ID] > 0
    has_bus     = cls_cnt[BUS_ID] > 0

    repeat = 3 if (has_motor or has_bicycle or has_bus) else 1
    for _ in range(repeat):
        train_list.append(img_path)

val_list = [m["img_path"] for m in val_meta]

print("train_list len:", len(train_list))
print("val_list len:", len(val_list))

TRAIN_TXT = DATA_ROOT / "train.txt"
VAL_TXT   = DATA_ROOT / "val.txt"

with open(TRAIN_TXT, "w", encoding="utf-8") as f:
    for p in train_list:
        f.write(p + "\n")

with open(VAL_TXT, "w", encoding="utf-8") as f:
    for p in val_list:
        f.write(p + "\n")

print("Đã ghi:", TRAIN_TXT)
print("Đã ghi:", VAL_TXT)





Tổng ảnh dataset: 24721
Tổng bbox theo lớp:
car     : 16008
truck   : 7640
person  : 36961
bus     : 1527
bicycle : 2178
motor   : 87244
Số ảnh train: 22249, val: 2472
train_list len: 50651
val_list len: 2472
Đã ghi: D:\Learn\Year4\KLTN\Dataset\traffic_yolo\train.txt
Đã ghi: D:\Learn\Year4\KLTN\Dataset\traffic_yolo\val.txt


In [8]:
import cv2

bad = []
for i, line in enumerate(open(TRAIN_TXT, "r", encoding="utf-8")):
    if i >= 200:  # kiểm tra 200 ảnh đầu
        break
    p = line.strip()
    img = cv2.imread(p)
    if img is None:
        bad.append(p)

print("Ảnh hỏng:", len(bad))
if bad:
    print("Ví dụ:", bad[:5])


Ảnh hỏng: 0


In [9]:

# %% [markdown]
# # 4. Tạo data_traffic.yaml

# %%
DATA_YAML = DATA_ROOT / "data_traffic.yaml"

yaml_lines = [
    f"path: {DATA_ROOT}",
    f"train: {TRAIN_TXT}",
    f"val: {VAL_TXT}",
    "",
    "names:",
]
for i, name in enumerate(TARGET_CLASSES):
    yaml_lines.append(f"  {i}: {name}")

with open(DATA_YAML, "w", encoding="utf-8") as f:
    f.write("\n".join(yaml_lines))

print("Đã tạo data_traffic.yaml:")
print(DATA_YAML.read_text())


Đã tạo data_traffic.yaml:
path: D:\Learn\Year4\KLTN\Dataset\traffic_yolo
train: D:\Learn\Year4\KLTN\Dataset\traffic_yolo\train.txt
val: D:\Learn\Year4\KLTN\Dataset\traffic_yolo\val.txt

names:
  0: car
  1: truck
  2: bus
  3: motor
  4: bicycle
  5: person


In [10]:
!pip uninstall -y ultralytics
!pip install ultralytics==8.3.17


Found existing installation: ultralytics 8.3.17
Uninstalling ultralytics-8.3.17:
  Successfully uninstalled ultralytics-8.3.17
  Using cached ultralytics-8.3.17-py3-none-any.whl.metadata (34 kB)
Using cached ultralytics-8.3.17-py3-none-any.whl (876 kB)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import urllib.request

url = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11s.pt"
urllib.request.urlretrieve(url, "yolo11s.pt")

import os
print(os.path.exists("yolo11s.pt"))
print(os.getcwd())


True
d:\Learn\Year4\KLTN


In [2]:
# %% [markdown]
# # 5. Cài Ultralytics & train YOLOv11s

# %%
from ultralytics import YOLO

model = YOLO("yolo11s.pt")
DATA_YAML = DATA_ROOT / "data_traffic.yaml"
run_name = "yolo11s_traffic"
project_dir = DATA_ROOT / "runs"

results = model.train(
    data=str(DATA_YAML),
    imgsz=640,
    epochs=100,        # chỉnh được
    batch=16,         # tuỳ GPU
    name=run_name,
    project=str(project_dir),
    workers=2,
    patience=15,
)

print("Train xong.")


New https://pypi.org/project/ultralytics/8.3.235 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.17  Python-3.10.18 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: task=detect, mode=train, model=yolo11s.pt, data=D:\Learn\Year4\KLTN\Dataset\traffic_yolo\data_traffic.yaml, epochs=100, time=None, patience=15, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=2, project=D:\Learn\Year4\KLTN\Dataset\traffic_yolo\runs, name=yolo11s_traffic2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=Fals

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/duong0410/d-learn-year4-kltn-dataset-traffic-yolo-runs/4611e947444d4bf28e3bfbe0afbbb20d

COMET INFO: Couldn't find a Git repository in 'd:\\Learn\\Year4\\KLTN' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
COMET WARNING: Unknown error exporting current conda environment
COMET WARNING: Unknown error retrieving Conda package as an explicit file
COMET WARNING: Unknown error retrieving Conda information


Freezing layer 'model.23.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks with YOLO11n...
AMP: checks passed 


train: Scanning D:\Learn\Year4\KLTN\Dataset\traffic_yolo\labels\all... 50651 images, 0 backgrounds, 0 corrupt: 100%|██████████| 50651/50651 [01:59<00:00, 423.92it/s]


train: New cache created: D:\Learn\Year4\KLTN\Dataset\traffic_yolo\labels\all.cache


...

                   all       2472      15124        0.9       0.79      0.863      0.702



100 epochs completed in 20.772 hours.
Optimizer stripped from D:\Learn\Year4\KLTN\Dataset\traffic_yolo\runs\yolo11s_traffic2\weights\last.pt, 19.2MB
Optimizer stripped from D:\Learn\Year4\KLTN\Dataset\traffic_yolo\runs\yolo11s_traffic2\weights\best.pt, 19.2MB

Validating D:\Learn\Year4\KLTN\Dataset\traffic_yolo\runs\yolo11s_traffic2\weights\best.pt...
Ultralytics 8.3.17  Python-3.10.18 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11s summary (fused): 238 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 78/78 [00:15<00:00,  5.03it/s]


                   all       2472      15124      0.892      0.797      0.868      0.705
                   car        756       1460      0.892      0.816      0.884      0.734
                 truck        547        749      0.945      0.903      0.955      0.892
                   bus        136        197      0.853      0.766      0.831      0.692
                 motor       1467       8516      0.964      0.981      0.991      0.883
               bicycle        174        260      0.844      0.708      0.802      0.536
                person        930       3942      0.854      0.609      0.743      0.495
Speed: 0.1ms preprocess, 2.1ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to D:\Learn\Year4\KLTN\Dataset\traffic_yolo\runs\yolo11s_traffic2


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\OS\\AppData\\Local\\Temp\\tmp22mn9tnn\\tmpm7l2tapa'

### 6. Đóng gói best.pt → Kaggle Dataset
* Đoạn này chỉ để khi chạy ngầm trên Kaggle (save version) thì có thể lấy weights để lưu sang thành Dataset *

* Lấy đúng user kaggle của người chạy nhé


In [ ]:
# %% [markdown]
# # 6. Đóng gói best.pt → Kaggle Dataset
## Đoạn này chỉ để khi chạy ngầm trên Kaggle (save version) thì có thể lấy weights để lưu sang thành Dataset
## Lấy đúng user kaggle của người chạy nhé
# %%


### đổi user name sang của người chạy chương trình - account của người hiện tại

import json
from pathlib import Path
import glob
import subprocess

best_pt_candidates = list(project_dir.glob(f"detect/{run_name}/weights/best.pt"))
if not best_pt_candidates:
    best_pt_candidates = list(project_dir.rglob("best.pt"))

if not best_pt_candidates:
    raise FileNotFoundError("Không tìm thấy best.pt trong thư mục runs.")

best_pt = best_pt_candidates[0]
print("best.pt:", best_pt)

KAGGLE_DS_DIR = Path("/kaggle/working/yolo11s_traffic_weights")
KAGGLE_DS_DIR.mkdir(parents=True, exist_ok=True)

copy2(best_pt, KAGGLE_DS_DIR / "best.pt")

last_candidates = list(project_dir.glob(f"detect/{run_name}/weights/last.pt"))
if last_candidates:
    copy2(last_candidates[0], KAGGLE_DS_DIR / "last.pt")

dataset_title = "YOLOv11s traffic weights"


PACKAGE_DIR = Path("/kaggle/working/yolo11s_traffic_weights")
assert PACKAGE_DIR.exists(), "PACKAGE_DIR chưa tồn tại, hãy chạy cell đóng gói weights trước."

metadata = {
    "title": "YOLOv11s traffic weights",  # <= 50 ký tự là an toàn
    "id": "caochung/yolo11s_traffic_weights",  # username/dataset-slug
    #  ^| subtitle: từ 20 đến 80 ký tự
    "subtitle": "YOLOv11s weights on traffic data",
    "licenses": [
        {"name": "CC0-1.0.1"}
    ],
    "description": (
        "Weights for YOLOv11 re-trained on \n Ms. COCO and UA-DETRACT dataset"
    )
}

meta_path = PACKAGE_DIR / "dataset-metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("\./ Đã ghi lại metadata:", meta_path)
print(meta_path.read_text(encoding="utf-8"))


print("📦 Thư mục package:", PACKAGE_DIR)

cmd_create = [
    "kaggle", "datasets", "create",
    "-p", str(PACKAGE_DIR),
    "-r", "zip"
]

print("🚀 Thử tạo dataset...")
proc = subprocess.run(cmd_create, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(proc.stdout)